# 2. RAG Revision

In [ ]:
from src import FaqHttpLoader, RAGBase, OpenRouterClient
from minsearch import Index 

loader = FaqHttpLoader()
documents = loader.load()

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)
index.fit(documents)

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=OpenRouterClient(),
    llm_model="openrouter/owl-alpha",
    instructions=instructions,
)

Testing it

In [ ]:
assistant.rag('How do I run Docker on Windows?')

In [ ]:
assistant.rag('How do I run ducker on windows?')

# 4. Function Calling [Ollama]

In [ ]:
import requests
llm_model = "granite4.1:8b"

prompt = 'I just discovered the course. Can I join it?'
url = "http://localhost:11434/api/chat"
payload = {
    "model": llm_model,
    "messages": [
        {"role": "user", "content": prompt},
    ],
    "stream": False,
    "keep_alive": 0, # 0 = unload immediately
}
response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()
data = response.json()
print(data["message"]["content"])

Defining the tool

In [ ]:
from pprint import pprint

# Correct format for Ollama /api/chat
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query text to look up in the course FAQ."
                }
            },
            "required": ["query"]
        }
    }
}

developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.
If you look up information, use FAQ search.
""".strip()
user_prompt = 'I just discovered the course. Can I join it?'

chat_messages = [
    {'role': 'developer', 'content': developer_prompt},
    {'role': 'user', 'content': user_prompt}
]

payload = {
    "model": llm_model,
    "messages": chat_messages,
    "stream": False,
    "keep_alive": 0, # 0 = unload immediately
    "tools": [search_tool],   # <-- add here
}
response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()
data = response.json()
msg = data["message"]
pprint(msg)

Executing the function and sending the result back

In [ ]:
import json

if msg.get("tool_calls"):
    print("Tool calls:")
    pprint(msg["tool_calls"])

tc = msg["tool_calls"][0]
fn_name = tc["function"]["name"]
fn_args = tc["function"]["arguments"]

# Выполняем поиск
results = assistant.search(**fn_args)
result_json = json.dumps(results, indent=2)

print(f"results found: {len(results)}")
# print("\nresult_json:"+"\n"+result_json)

Add the model's output to the conversation history. Then we add the tool result:

In [ ]:
chat_messages.append(msg)

chat_messages.append({
    "type": "function_call_output",
    'call_id': tc['id'],
    'output': result_json,
})
# The call_id links the tool result to the specific function call the model requested. 
# If the model makes multiple function calls in one turn, each one gets its own call_id.

In [ ]:
for m in chat_messages:
    print(m,"\n----------\n")

Asking the model again

In [ ]:
payload["messages"] = chat_messages
response = requests.post(url, json=payload, timeout=120)
response.raise_for_status()
data = response.json()
msg = data["message"]
pprint(msg)

In [ ]:
pprint(msg["content"])

# 5. The Agentic Loop [OpenRouter]

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",                          # на верхнем уровне, не внутри "function"
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {                            # тоже на верхнем уровне
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.
If you look up information, use FAQ search.
""".strip()
user_prompt = 'I just discovered the course. Can I join it?'

chat_messages = [
    {'role': 'developer', 'content': developer_prompt},
    {'role': 'user', 'content': user_prompt}
]

In [ ]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)

response = openai_client.responses.create(
    model='openrouter/owl-alpha', # openrouter/owl-alpha, z-ai/glm-5.1 , openai/gpt-oss-120b:free
    input=chat_messages,
    tools=[search_tool],
)

In [ ]:
response.output_text

In [ ]:
import json
from openai.types.responses import ResponseFunctionToolCall

call = next(
    item for item in response.output
    if isinstance(item, ResponseFunctionToolCall)
)
args = json.loads(call.arguments)
search = assistant.search
result = search(**args)
result_json = json.dumps(result, indent=2)

chat_messages.extend(response.output)
chat_messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

response2 = openai_client.responses.create(
    model='openrouter/owl-alpha',
    input=chat_messages,
    tools=[search_tool],
)
response2.output_text


A stronger developer prompt

In [ ]:
developer_prompt = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches if needed. Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

chat_messages = [
    {'role': 'developer', 'content': developer_prompt},
    {'role': 'user', 'content': user_prompt}
]

In [ ]:
chat_messages

A generic function-call helper

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)
    f_name = call.name
    f = globals()[f_name]
    result = f(**args)
    result_json = json.dumps(result, indent=2)
    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

Processing one response

In [ ]:
response = openai_client.responses.create(
    model='openrouter/owl-alpha',
    input=chat_messages,
    tools=[search_tool],
)

has_function_calls = False

for entry in response.output:
    chat_messages.append(entry)

    if entry.type == 'message':
        print(entry.content[0].text)

    if entry.type == 'function_call':
        print('function_call:', entry.name, entry.arguments)
        result = make_call(entry)
        chat_messages.append(result)
        has_function_calls = True

The full agent loop

In [ ]:
while True:
    response = openai_client.responses.create(
        model='openrouter/owl-alpha',
        input=chat_messages,
        tools=[search_tool],
    )

    chat_messages.extend(response.output)
    has_function_calls = False

    for entry in response.output:
        if entry.type == 'message':
            print(entry.content[0].text)

        if entry.type == 'function_call':
            print('function_call:', entry.name, entry.arguments)
            result = make_call(entry)
            chat_messages.append(result)
            has_function_calls = True

    if not has_function_calls:
        break

Wrapping it in a function

In [ ]:
def agent_loop(question, model='arcee-ai/trinity-large-thinking:free'):
    chat_messages = [
        {'role': 'developer', 'content': developer_prompt},
        {'role': 'user', 'content': question}
    ]

    for i in range(10):
        print(f"\n--- iteration {i} ---")
        response = openai_client.responses.create(
            model=model,
            input=chat_messages,
            tools=[search_tool],
        )
        
        # print("output:", response.output)
        # print("output_text:", response.output_text)
        print("status:", response.status)

        # Исправление: model_dump() чтобы не было строк в истории
        chat_messages.extend(item.model_dump() for item in response.output)
        has_function_calls = False

        for entry in response.output:
            if entry.type == 'message':
                print(entry.content[0].text)
            if entry.type == 'function_call':
                print('function_call:', entry.name, entry.arguments)
                result = make_call(entry)
                chat_messages.append(result)
                has_function_calls = True

        if not has_function_calls:
            break

In [ ]:
# baidu/cobuddy:free  arcee-ai/trinity-large-thinking:free  nvidia/nemotron-3-super-120b-a12b:free
# agent_loop('How do I run ducker on windows?')
agent_loop(
    'How do I run ducker on windows?',
    model='arcee-ai/trinity-large-thinking:free'
    )


In [ ]:
agent_loop('I just discovered the course. Can I still join it?')

# 6. ToyAIKit framework

In [ ]:
import dotenv

dotenv.load_dotenv(override=True)

In [ ]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

# Registering the tool
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

Create the chat interface and the runner:

In [ ]:
chat_interface = IPythonChatInterface()

# create the LLM client
openrouter_client = OpenAI(
    api_key=os.environ.get("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
)
llm_client = OpenAIClient(
    model="arcee-ai/trinity-large-thinking:free",   # модель которая поддерживает tools
    client=openrouter_client,
)

# create the runner
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=developer_prompt,
    chat_interface=chat_interface,
    llm_client=llm_client
)

Running one prompt

In [ ]:
callback = DisplayingRunnerCallback(chat_interface)
messages = runner.loop(
    prompt='How do I run ducker on Windows?',
    callback=callback
    )

Interactive chat

In [ ]:
runner.run()